# 5. Foundation-model embeddings and the MedGemma prompt

Reproduces **Section 3.4** (how the two representations are built) and **Section 4.6** (how much
the prompt matters):

| output | paper |
|---|---|
| four prompt formulations, standalone AUC | `tab:prompts` |
| the numeric prompt's single-field AUCs | Section 4.6 |
| the answer distribution saturating and then flipping | Section 3.4 and Section 4.6 |
| BiomedCLIP on the T1ce crop and on the enhancement map | Section 4.5, 0.797 and 0.855 |

The table cells run from `data_cache/`. The generation cells need the BraTS copy plus the
MedGemma and BiomedCLIP weights, and are guarded.

## What each encoder is shown

Both read one axial slice per case, the one with the largest **predicted** tumor area, so the
choice never touches a reference mask.

MedGemma gets two panels side by side: the T1ce crop and the enhancement map of the same region.
Both are windowed in standard deviations of healthy brain, not by each crop's own percentiles.
That detail matters more than it looks: per-crop normalization stretches a non-enhancing tumor's
noise across the full range, so a low-grade and a high-grade case come out equally bright and the
enhancement panel stops carrying the signal it exists for.

BiomedCLIP gets a single crop of the same region and its image embedding is taken directly.

Both embeddings are reduced with PCA to sixteen components, fit inside the training folds only as
part of the cross-validation pipeline. The volumetric features are standardized but not reduced.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, encoders, tables

print("MedGemma:", config.MEDGEMMA_ID)
print("BiomedCLIP:", config.BIOMEDCLIP_ID)
print("panels:", f"{encoders.PANEL_SIZE} x {encoders.PANEL_SIZE} each, side by side")
print("T1ce window, in standard deviations of healthy brain:", encoders.T1CE_WINDOW)
print("enhancement window:", encoders.ENHANCEMENT_WINDOW)

## The four prompts

All four ask for six fields about the tumor. They differ in what surrounds that request.

- **Baseline (categorical)** asks for the six fields and nothing else.
- **Numeric (percentages)** replaces the categories with integers from 0 to 100, on the reasoning
  that a category can saturate at one value and a percentage cannot.
- **Few-shot** adds a written calibration block: what avid, mild and absent enhancement look like
  on the enhancement panel, and a statement that many gliomas do not enhance, that "none" is a
  common and expected answer, and that the tumor should be judged from what is visible rather than
  assumed to be high-grade. There are no image exemplars, so no case label can leak into the
  features and the cohort stays at 369.
- **Few-shot + reasoning** keeps that calibration and asks for a short reasoning step before the
  answer. Only the tokens after the marker are pooled, so the vector represents the conclusion and
  not the reasoning.

The numeric, few-shot and reasoning prompts open with the same description of the two panels. As
first submitted, that description called normal brain on the enhancement map mid-grey. On the
panel the model is actually shown, the window puts healthy brain at a quarter of the grey range,
so normal brain and the empty background are dark grey. The three prompts correct that one clause
and change nothing else (`encoders.SUBMISSION_CLAUSE` and `encoders.CORRECTED_CLAUSE` hold both
versions); the baseline prompt never contained it. The few-shot prompt, this correction
included, was written after earlier results were known, as the paper's limitations state.

In [ ]:
for key, prompt in encoders.PROMPTS.items():
    print("=" * 78)
    print(f"{prompt['label']}   [{key}]")
    print("=" * 78)
    print(prompt["system"])
    print(f"\nuser turn: {prompt['user']}")
    print(f"pooled from: {'after ' + prompt['marker'] if prompt['marker'] else 'the whole answer'}\n")

In [ ]:
print("submission clause:", encoders.SUBMISSION_CLAUSE)
print("corrected clause: ", encoders.CORRECTED_CLAUSE)
display(pd.DataFrame({
    key: {"submission clause": encoders.SUBMISSION_CLAUSE in p["system"],
          "corrected clause": encoders.CORRECTED_CLAUSE in p["system"]}
    for key, p in encoders.PROMPTS.items()}).T)

## What the model actually answers

The baseline prompt saturates: "avid" for all 369 cases, and high-grade for 364 of them, against
76 true low-grade cases. The calibration breaks the saturation and overshoots in the other
direction: "none" for 306 of 369 and high-grade for only 19. Asking for a reasoning step first
talks the model back towards its prior, to 147 "avid" and 139 high-grade.

That is worth being precise about, because the literal answer is not what the few-shot embedding
is scored on. The answer stays a coarse grader either way; what the calibration changes is the
perception encoded in the hidden state, and that is what the table below measures.

In [ ]:
distributions = {}
for field in ("Enhancement", "Assessment"):
    for key, prompt in encoders.PROMPTS.items():
        path = config.MEDGEMMA_ANSWERS[key]
        counts = encoders.answer_distribution(path, field, top=10, marker=prompt["marker"])
        distributions[(field, prompt["label"])] = counts
answers = pd.DataFrame(distributions).fillna(0).astype(int)
display(answers)

expected = {
    ("Enhancement", "Baseline (categorical)"): {"avid": 369, "mild": 0, "none": 0},
    ("Enhancement", "Few-shot"): {"none": 306, "mild": 59, "avid": 4},
    ("Enhancement", "Few-shot $+$ reasoning"): {"none": 74, "mild": 148, "avid": 147},
    ("Assessment", "Baseline (categorical)"): {"high-grade": 364, "low-grade": 5},
    ("Assessment", "Few-shot"): {"high-grade": 19, "low-grade": 350},
    ("Assessment", "Few-shot $+$ reasoning"): {"high-grade": 139, "low-grade": 230},
}
print("answer counts match:", all(int(answers.loc[value, column]) == n
                                  for column, counts in expected.items()
                                  for value, n in counts.items()))

## Generating the embeddings

Guarded, because this is the expensive path: roughly 7 seconds per case for the categorical
prompts and about 12 for the reasoning one, plus the model download. The caches these produce ship
with the repository.

In [ ]:
RUN_INFERENCE = False   # needs the BraTS copy, the cross-fitted masks and the model weights

# The generation functions skip every case already in their output files, so pointed at the
# shipped caches they would do nothing. They write to a fresh directory instead, which leaves
# data_cache/ untouched and gives a rerun to compare against it.
REGENERATED = config.DATA_ROOT / "regenerated"

if RUN_INFERENCE:
    for key in encoders.PROMPTS:
        encoders.run_medgemma(key, answers_path=REGENERATED / config.MEDGEMMA_ANSWERS[key].name,
                              hidden_path=REGENERATED / config.MEDGEMMA_HIDDEN[key].name)
    for which, cache in (("T1ce", config.BIOMEDCLIP_T1CE), ("enhancement", config.BIOMEDCLIP_ENH)):
        encoders.embed_biomedclip_all(which, out_path=REGENERATED / cache.name)
    # MedGemma's own grade-token confidence, teacher-forced on the saved baseline answers.
    encoders.run_native_confidence(out_path=REGENERATED / config.MEDGEMMA_NATIVE_CONF.name)

## tab:prompts

Each prompt's description embedding scored on its own, under the pooled out-of-fold protocol, the
same one `tab:errcorr` uses. Expected:

| Prompt | Standalone AUC | $\Delta$ vs. baseline | $\Delta$ vs. Vol. |
|---|---|---|---|
| Baseline (categorical) | 0.786 | --- | -0.139 |
| Numeric (percentages) | 0.763 | -0.023 | -0.162 |
| Few-shot | **0.804** | +0.018 | -0.121 |
| Few-shot + reasoning | 0.618 | -0.168 | -0.307 |

The reading: the calibration gives the strongest standalone embedding, +0.018 over the baseline
prompt. Asking for numeric percentages lowers it slightly, and the estimated values themselves
carry no grade signal (the single-field AUCs below sit at chance). Adding a reasoning step
**hurts**, which is consistent with the failure being perceptual rather than logical: reasoning
talks the model back towards its prior, as the answer counts above show.

Even the strongest prompt stays well below the volumetric baseline of 0.925 on its own, so the
prompt matters only through the fusion in notebook 6, never as a grader in itself.

Note on protocol, since it changes the third decimal: these are pooled out-of-fold AUCs, computed
once over the concatenated fold predictions. Averaging the five per-fold AUCs instead moves the
third decimal; the cell after the table prints both. The caption states which one it is, and this
is it.

In [ ]:
blocks, y, cases = cohort.load_blocks()
cohort.inject_encoder_blocks(blocks, cases, keys=(
    "medgemma_fewshot", "medgemma_numeric", "medgemma_fewshot_cot", "biomedclip_enh"))

In [ ]:
prompts = tables.table_prompts(blocks, y)
display(prompts)

published = {
    "Baseline (categorical)": ("0.786", "---", "-0.139"),
    "Numeric (percentages)": ("0.763", "-0.023", "-0.162"),
    "Few-shot": ("0.804", "+0.018", "-0.121"),
    "Few-shot $+$ reasoning": ("0.618", "-0.168", "-0.307"),
}
check = tables.check_printed(prompts, {
    row: dict(zip(["Standalone AUC", "$\\Delta$ vs. baseline", "$\\Delta$ vs. Vol."], values))
    for row, values in published.items()})
display(check)
print("all cells match the paper:", bool(check["match"].all()))

In [ ]:
from src.evaluation import block_auc, out_of_fold
from sklearn.metrics import roc_auc_score

protocols = pd.DataFrame([
    {"Prompt": label,
     "pooled out-of-fold": round(roc_auc_score(y, out_of_fold(blocks, [key], y)), 4),
     "mean of the five folds": round(block_auc(blocks, [key], y), 4)}
    for label, key in tables.PROMPTS]).set_index("Prompt")
display(protocols)

### What the numeric answers carry on their own

Each of the six percentages the numeric prompt writes, used directly as a grade score. Expected:
0.514, 0.542, 0.540, 0.510, 0.522 and 0.534, which is chance. Whatever the numeric prompt's hidden
state encodes, it is not in the numbers it writes.

In [ ]:
single = encoders.single_field_aucs(config.MEDGEMMA_ANSWERS["numeric"], cases, y)
display(single)
expected = {"EnhancingPercent": 0.514, "NecroticPercent": 0.542, "EdemaPercent": 0.540,
            "MarginClarity": 0.510, "MassEffect": 0.522, "HighGradeConfidence": 0.534}
display(tables.compare_to_paper(single, "Single-field AUC", expected, tolerance=0))

## The two BiomedCLIP inputs

The slice is fixed and only the image changes, so this isolates what the encoder is shown. The
T1ce crop gives **0.797**; the enhancement map, which is the same signal MedGemma reads, gives
**0.855**.

The better input on its own does not become a better complement. In fusion neither adds more than
+0.002 (notebook 6), and the enhancement-map errors are the most correlated with the volumetric
ones (0.483 within grade): the enhancement map is what the volumetric features already measure.

In [ ]:
inputs = pd.DataFrame([
    {"BiomedCLIP input": "T1ce crop",
     "Standalone AUC": round(roc_auc_score(y, out_of_fold(blocks, ["biomedclip_t1ce"], y)), 3)},
    {"BiomedCLIP input": "enhancement map (T1ce - T1)",
     "Standalone AUC": round(roc_auc_score(y, out_of_fold(blocks, ["biomedclip_enh"], y)), 3)},
]).set_index("BiomedCLIP input")
display(inputs)
display(tables.compare_to_paper(inputs, "Standalone AUC",
                                {"T1ce crop": 0.797, "enhancement map (T1ce - T1)": 0.855},
                                tolerance=0))

Both are above the 0.65 threshold fixed in advance as the level below which the experiment would
have been abandoned, and both stay below the volumetric baseline of 0.925.